In [1]:
!pip install ./semseg/
# !./download_model.sh

Processing ./semseg
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mit_semseg: filename=mit_semseg-1.0.0-py3-none-any.whl size=47047 sha256=a8e151c2c7f97deaa1e22e31776f7d15b7b9f770dfa8b8fca122dd6739939a0b
  Stored in directory: /tmp/pip-ephem-wheel-cache-90gway9b/wheels/c1/c2/d9/93afd44a3689c102e8e64430bb88e7e32cef30e3fce6404e48
Successfully built mit_semseg
  Attempting uninstall: mit_semseg
    Found existing installation: mit_semseg 1.0.0
    Uninstalling mit_semseg-1.0.0:
      Successfully uninstalled mit_semseg-1.0.0


In [2]:
# System libs
import os, csv, torch, numpy, scipy.io, PIL.Image, torchvision.transforms
# Our libs
from mit_semseg.models import ModelBuilder, SegmentationModule
from mit_semseg.utils import colorEncode

colors = scipy.io.loadmat('data/color150.mat')['colors']
names = {}
with open('data/object150_info.csv') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        names[int(row[0])] = row[5].split(";")[0]

def visualize_result(img, pred, index=None):
    # filter prediction class if requested
    if index is not None:
        pred = pred.copy()
        pred[pred != index] = -1
        print(f'{names[index+1]}:')
        
    # colorize prediction
    pred_color = colorEncode(pred, colors).astype(numpy.uint8)

    # aggregate images and save
    im_vis = numpy.concatenate((img, pred_color), axis=1)
    display(PIL.Image.fromarray(im_vis))

In [15]:
def train(num_epoch=20, start_epoch=0, epoch_iters=5000, cfg='config/ade20k-resnet50dilated-ppm_deepsup.yaml', prune=True, ratio=1, opts='', silent=False):
    if prune:
        if silent:
            !python3 ./utils/train_prune.py --gpus 0 --cfg {cfg} --ratio {ratio} TRAIN.num_epoch {num_epoch} TRAIN.start_epoch {start_epoch} TRAIN.epoch_iters {epoch_iters} {opts} > /dev/null
        else:
            !python3 ./utils/train_prune.py --gpus 0 --cfg {cfg} --ratio {ratio} TRAIN.num_epoch {num_epoch} TRAIN.start_epoch {start_epoch} TRAIN.epoch_iters {epoch_iters} {opts}
    else:
        if silent:
            !python3 ./utils/train.py --gpus 0 --cfg {cfg} --ratio {ratio} TRAIN.num_epoch {num_epoch} TRAIN.start_epoch {start_epoch} TRAIN.epoch_iters {epoch_iters} {opts} > /dev/null
        else:
            !python3 ./utils/train.py --gpus 0 --cfg {cfg} --ratio {ratio} TRAIN.num_epoch {num_epoch} TRAIN.start_epoch {start_epoch} TRAIN.epoch_iters {epoch_iters} {opts}

In [4]:
def evaluate(cfg='config/ade20k-resnet50dilated-ppm_deepsup.yaml', opts="", file=None):
    if file:
        !python3 ./utils/eval_multipro.py --gpus 0 --cfg {cfg} {opts} > {file}
    else:
        !python3 ./utils/eval_multipro.py --gpus 0 --cfg {cfg} {opts}

In [5]:
def test(path, cfg='config/ade20k-resnet50dilated-ppm_deepsup.yaml' , opts=""):
    !python3 -u ./utils/test.py --imgs {path} --gpu 0 --cfg {cfg} {opts}

# Punning the model

In [6]:
net_encoder = ModelBuilder.build_encoder(
    arch='resnet50dilated',
    fc_dim=2048,
    weights='ckpt/ade20k-resnet50dilated-ppm_deepsup/encoder_epoch_30.pth')
net_decoder = ModelBuilder.build_decoder(
    arch='ppm_deepsup',
    fc_dim=2048,
    num_class=150,
    weights='ckpt/ade20k-resnet50dilated-ppm_deepsup/decoder_epoch_30.pth',
    use_softmax=True)

crit = torch.nn.NLLLoss(ignore_index=-1)
module = SegmentationModule(net_encoder, net_decoder, crit)
module.eval()
module.cuda()

Loading weights for net_encoder


FileNotFoundError: [Errno 2] No such file or directory: 'ckpt/ade20k-resnet50dilated-ppm_deepsup/encoder_epoch_30.pth'

In [7]:
def sparsity(module):
    print(
    "Sparsity in conv1.weight: {:.2f}%".format(
        100. * float(torch.sum(module.weight == 0))
        / float(module.weight.nelement())
    )
)

In [ ]:
pil_to_tensor = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(
        mean=[0.485, 0.456, 0.406], # These are RGB mean+std values
        std=[0.229, 0.224, 0.225])  # across a large photo dataset.
])
pil_image = PIL.Image.open('ADE_val_00001519.jpg').convert('RGB')
img_original = numpy.array(pil_image)
img_data = pil_to_tensor(pil_image)
singleton_batch = {'img_data': img_data[None].cuda()}
output_size = img_data.shape[1:]

In [ ]:
# Run the segmentation at the highest resolution.
with torch.no_grad():
    scores = module(singleton_batch, segSize=output_size)
    
# Get the predicted scores for each pixel
_, pred = torch.max(scores, dim=1)
pred = pred.cpu()[0].numpy()
visualize_result(img_original, pred)

In [ ]:
# Top classes in answer
predicted_classes = numpy.bincount(pred.flatten()).argsort()[::-1]
for c in predicted_classes[:15]:
    visualize_result(img_original, pred, c)

In [16]:
# test_ratios = [0.3, 0.5, 0.7]
r = 0.5
iters = 1000
start = 20
mid = 5
# for r in test_ratios:
#     print(f"Testing ratio: {r}")
#     train(num_epoch=start+mid, start_epoch=start, epoch_iters=iters, ratio=r)
#     evaluate(opts=f'VAL.checkpoint epoch_{start+mid}_{r}.pth', file=f"./eval/eval_{r}.txt")
#     # train(num_epoch=start+2*mid, start_epoch=start+mid, epoch_iters=iters, prune=False)
#     # evaluate(opts=f'VAL.checkpoint epoch_{start+2*mid}_{r}.pth', file=f"./eval/eval_{r}_fine.txt")
print(f"Testing ratio: {r}")
train(num_epoch=start+2*mid, start_epoch=start+mid, epoch_iters=iters, prune=False, ratio=r)
evaluate(opts=f'VAL.checkpoint epoch_{start+2*mid}_{r}.pth', file=f"./eval/eval_{r}_fine.txt")
print("Done!")

Testing ratio: 0.5
/home/mjmaza/cv-project/./utils/train.py:209: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  assert LooseVersion(torch.__version__) >= LooseVersion('0.4.0'), \
[2025-05-13 09:27:13,399 INFO train.py line 244 16982] Loaded configuration file config/ade20k-resnet50dilated-ppm_deepsup.yaml
[2025-05-13 09:27:13,399 INFO train.py line 245 16982] Running with config:
DATASET:
  imgMaxSize: 1000
  imgSizes: (300, 375, 450, 525, 600)
  list_train: ./data/training.odgt
  list_val: ./data/validation.odgt
  num_class: 150
  padding_constant: 8
  random_flip: True
  root_dataset: ./data/
  segm_downsampling_rate: 8
DIR: ckpt/ade20k-resnet50dilated-ppm_deepsup
MODEL:
  arch_decoder: ppm_deepsup
  arch_encoder: resnet50dilated
  fc_dim: 2048
  weights_decoder: 
  weights_encoder: 
TEST:
  batch_size: 1
  checkpoint: epoch_20.pth
  result: ./
TRAIN:
  batch_size_per_gpu: 2
  beta1: 0.9
  deep_sup_scale: 0.4
  disp_iter: 20
  epoch_ite